# **DINO enhanced visualisation**

## **1. Importing**

In [8]:
import torch
import numpy as np
import torchvision
from transformers import AutoImageProcessor, AutoModel
from torch.utils.data import DataLoader
import h5py
import random
import kagglehub
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms
import matplotlib.pyplot as plt

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### **1.2 DINO FLAGSHIP MODEL 40 LAYERS WOHOO**

In [10]:
model_name = "facebook/dinov3-vit7b16-pretrain-lvd1689m"

print(f"Loading flagship model: {model_name}")
print(f"Targeting Architecture: 40 Layers, 4096 Embedding Dim")

try:
    # 3. Load Model with Memory Optimization
    model = AutoModel.from_pretrained(
        model_name,
        torch_dtype=dtype,
        low_cpu_mem_usage=True
    ).to(device).eval()

    # 4. Load Processor
    processor = AutoImageProcessor.from_pretrained(model_name)
    transform = lambda img: processor(images=img, return_tensors="pt")['pixel_values'].squeeze()

    # 5. Verify Layer Count
    # Depending on the specific Hugging Face implementation, blocks might be in 'model.layers'
    if hasattr(model, "vision_model"):
        num_layers = len(model.vision_model.encoder.layers)
    else:
        # Fallback for flat architectures
        num_layers = len(model.layer)

    print(f"Success! Model loaded with {num_layers} layers.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Tip: If you run out of memory (OOM), try 'dinov3-vitl16' (Large, 24 layers) instead.")

Loading flagship model: facebook/dinov3-vit7b16-pretrain-lvd1689m
Targeting Architecture: 40 Layers, 4096 Embedding Dim


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Success! Model loaded with 40 layers.


## **2. Dataset**

### **2.1 CIFAR**

In [11]:
dataset_cifar10 = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
loader_cifar10 = DataLoader(dataset_cifar10, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of CIFAR10 samples (OOD - CIFAR-10): {len(dataset_cifar10)}")

Amount of CIFAR10 samples (OOD - CIFAR-10): 10000


In [12]:
dataset_cifar100 = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
loader_cifar100 = DataLoader(dataset_cifar100, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of CIFAR100 samples (OOD - CIFAR-100): {len(dataset_cifar100)}")

Amount of CIFAR100 samples (OOD - CIFAR-100): 10000


### **2.2 SVHN**

In [13]:
dataset_svhn = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform)
loader_svhn = DataLoader(dataset_svhn, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of SVHN samples (Test set): {len(dataset_svhn)}")

Amount of SVHN samples (Test set): 26032


### **2.3 FOOD101**

In [14]:
dataset_food101 = torchvision.datasets.Food101(root='./data', split='test', download=True, transform=transform)
loader_food101 = DataLoader(dataset_food101, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of Food101 samples (test set): {len(dataset_food101)}")

100%|██████████| 5.00G/5.00G [02:52<00:00, 29.0MB/s]


Amount of Food101 samples (test set): 25250


### **2.4 Pokémon**

In [15]:
path = kagglehub.dataset_download("thedagger/pokemon-generation-one")

print("Path to dataset files:", path)

100%|██████████| 2.15G/2.15G [00:13<00:00, 171MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/thedagger/pokemon-generation-one/versions/1


In [16]:
transform_kaggle = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

image_root_path = os.path.join(path, 'dataset')

dataset_pokemon_gen1 = ImageFolder(
    root=image_root_path,
    transform=transform_kaggle
)

loader_pokemon_gen1 = DataLoader(
    dataset_pokemon_gen1,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

print(f"\n✅ Success! Loaded {len(dataset_pokemon_gen1)} images.")
print(f"Found classes: {dataset_pokemon_gen1.classes[:5]}...")


✅ Success! Loaded 21314 images.
Found classes: ['Abra', 'Aerodactyl', 'Alakazam', 'Arbok', 'Arcanine']...
